# RepairScope-X Notebook 19 — BD3LM 0.5B self-repair

**Research objective:** Extend RepairScope-X beyond Dream and ALoDLM using a genuine small diffusion-LM family. This model is `dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-bd3lm-v0.1`, not a language-model simulation.

**What this experiment does**
1. Runs actual model-native diffusion generation on **24 frozen problems** (8 Countdown, 8 GSM8K, 8 SVAMP).
2. Selects up to **12 natural task failures** without using repair outcomes.
3. Compares four predeclared repair actions, with **two repair seeds** each (up to 96 repair trials, plus 24 baseline generations).
4. Evaluates strict answer, numeric answer, and task-aware success; verifies held-token preservation; records actual forward calls, wall time and GPU memory.
5. Produces per-action and per-dataset tables, problem-cluster bootstrap intervals, paired per-problem tests, and automatic ZIP export.

**Specific model architecture:** block discrete diffusion with prefix repair. BD3LM conditions: `tail25`, `tail50`, `tail75`, `global`. The observed prefix conditions new blocks; arbitrary interior remasking is **not** implemented.

**Important limitations:** The two new models share the Qwen2.5-Coder-0.5B architecture and dLLM training ecosystem; they are two distinct checkpoints and diffusion objectives, **not two independent foundation-model families**. Coder-specialized checkpoints may score poorly on arithmetic, so these trials are exploratory. This is *not* a multimodal experiment. The actual inference path cannot be GPU-tested by the notebook author here; a built-in live smoke test runs before the sweep.

**Hardware:** Google Colab GPU (T4 recommended). One notebook per independent Colab runtime. Each notebook has all benchmark prompts embedded and does not require earlier result uploads.

Benchmark payload SHA-256: `cc04b82051f62f8640b92c418571d5efd3fd89334bf4be126bfae6f508ff5328`.

Model card: https://huggingface.co/dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-bd3lm-v0.1

In [ ]:
!pip -q install "transformers==4.56.1" "accelerate==1.10.1" "huggingface-hub>=0.34,<1.0" "safetensors>=0.6" numpy pandas matplotlib

In [ ]:
import os,sys,random,time,hashlib,json,base64,zlib,math,re,ast as pyast,gc,platform,traceback,zipfile
from pathlib import Path
from collections import Counter,defaultdict
from datetime import datetime,timezone
import numpy as np,pandas as pd,torch,transformers
from huggingface_hub import model_info,snapshot_download
from transformers import AutoTokenizer,AutoModelForMaskedLM
print('Python',platform.python_version(),'torch',torch.__version__,'transformers',transformers.__version__)
assert torch.cuda.is_available(), 'Set Runtime > Change runtime type > T4 GPU before running.'
print('GPU',torch.cuda.get_device_name(0),'VRAM GB',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
assert torch.cuda.get_device_properties(0).total_memory/2**30 >= 7, 'Use a GPU with at least 7 GB VRAM.'
if torch.cuda.memory_allocated()/2**30>0.7: raise RuntimeError('GPU already holds a model. Restart runtime for a clean independent run.')
torch.manual_seed(3819);random.seed(3819);np.random.seed(3819)

In [ ]:
PAYLOAD_SHA="cc04b82051f62f8640b92c418571d5efd3fd89334bf4be126bfae6f508ff5328"
_encoded="eNq1WW2P27gR/iuEsUW+eBeWLPkNRYttkyAX3KXFbooULQofLdMWYUlUScqOcrj/3mdIyZad7a1tnIFssmtyhjPPPPO2+aVnVKUTYXqzf//SK7VaZCKfy2Vv1ktUVdil2hWzhGdyobmVqpgFk0E8iwa9fm/JLTfCdm/OV1qYFGcl11bSfTo9SNOJVnlJQv8wgtlUsKLKF0IbFk77DH94sWQTJr7yxGY1U0WCW4pBalklXsByvRaWBeEDexK20gXjbIs3loxradNcWJlAQQlTDN50GgW+djhkP7//4dPjj/PHT89f3j3N2B/963/6+QGmecW9GTQPfu2fj8bk9wcjiPosGHg0wuFZcNwMjcvAiMY3oEafDT0W8TlQDAe3wgKaLwIjvAE1kCPBuGHG4Bw44vBmeJDqiwAJbsCOYNTJlbNKxzC6GUGiy/CIb5IsYXQJGvHN0IgvRGMY/P5wDMGMsGFHcFa+RDdLl+jS8jG9AT2IH/El9eM+HN4KEFL9PSJrk082M6u5LGbxcHQMgjs8z/WfuLUs5QZWl4IXlWVlxgvr7sFBbpk0LB4M2ErAzUW9//6BsYCZ/1ZcC7ZSyjK1ajQYlvCC5XwjcJmtNc9N5xByP6yYtEBuIwzBe3qDIPbSp0dsUVkrtOcp26y/PzEiywzs0ewORE7VjuVVkrKlwlOIllO7gu8shVsHT//cLU9BGA9eQXw0GsfXIv6R5zBmUdWGjZi9N6nU1pscD/4Aj1ZA6HMqaiY4LE+UseyOuhP7sPcG5IIzJa+7do9eM3ocDMdXG63Sgi1cxIIQRiWKQM95wrVShel7Y3dCrlNZrBE5pCEG5gf2waVJAWOTDUhWMAHKZE2mUfysC0ujh8mCRfBvtRJaFIiqRibj3TVe0IIva4fTUmRyK3T9wL6QaihYaClWyNBM8C1MJJ0baRM6pXyVycZ9VqaqEH0X+0xam5GgwoFmz1ZsBZywXhhvS/AaXlp86xgIydbcg1WIypcmR1wxUBa+ORRsn3zxKPRbQS1yBIIA+g7BbiSj+LWEH4ejayP5meygT1ujeE4lk37K4WPN/so177OPvBDWV8CPQusaNQIAwc9oFs9GD5TDB49PVViibytwN0YlhTFJlXHri2amjHHveP30FOphlvn8lCuvIFH5QhZOQupWs6LMYUrzYi2ME6eUpzs5QyAmLoV8lKqakC61TES3nk5fze3o6tx+hkFauXpq5FdmJaU6fsh5UbMlr8AL6i90jJg7FGFpUmlie1Y7QZtqIdx53xcwkk1SmS1xaV/JCN0uZ8LgVc6Mr87+90g6gnTFcwkrtagK3OizvwuqudK0WAcjVoIJS5cwCwTsHjkA05ON8I0v5dnKd5WGMjhPeb6o4ISmbPog3hi2VqSsCXWIyCMVD3pT9JqlWhNQ/CVpnhl1rMIgtzVoauTSFQfesM0lamtTNxHaJ9qiSxE4GIBut/QKPMuOmser7AriyfVF2Igybbq1Fist4Te3SiNP2Y5TLFLh/iFYEZZUVUb4JKbKm4nEIhuY2oqmt6MImYo4Wqod9VXbqb3QTAX8829kOe43NqEX+R7mq6fQ65rh7SVNDL7yHaz11ZJI7vOjg73vagdN3qyuoq6b+yp07Bmutw7tlMaUdUczi6TBDIbbtM/sDjWB7f6f/V729DWn97SSHSHzqvUYM9d8CaKYo18ewLzXkjeKwquTF7UykeZNm71En5BVmyoDbHAoYutKwhbTEIUlQls82unRW6kySfXqHXX5RtQpipgB8MW6PfOq3NHo5MgrOZU66iOHF5tzQpbyGDcOBh+Z25jmOtOgFesUzvbc1c3voTgpoi+EwWx5XjZhSFAr7qej4VEo3IXzQvG4Ih6BTVScOPFK25rlgme+4+0o71C0jPDs+kuVbHAVNQ7Eyjj6GKWjRko/AVGR0byjC9DLpWIUw1sjkWE0geduVXlRC3OjPfipBeCpAA6vXSQm4RTztCoRXKfJaX5JAS0vslHCMaBOYtJhDtVyb8hv62l3jOXxKDueROdEIjj5TckFkXCMLJVffUCcVUa56uZPaRuUGa020eDo0PH0odsVMB3SPeskqMhkWdeVYHAWpybBtZ5gzdHAt6QBvJ2IFILvuuYaVbY4nHkL3XDEzYZ2OJ9+ja+8SUPkXxDvpY4kDp6/orqLQXwOBOPB1RC8o2WAOMSeavQ5Uwrq1CNqf4gaOObShHJOFOtMmtQv9i+cY1YpUKU7AXYw+ovt6OX0v6jWTyTC33xB79GQfw4ow8HVDD9QGAzpbFRKbTxLRp1Pc7WVh1C/YYnm3zA3YcbB30mqVIYPjcCW5Ut2rSq3j7Eg2u9wXgVpRtcEKXA43c/i7tVjUP1nDrJG1m0MreqLCRRPo2uxeltp2bSZR73hheHekc/iKw3kghqbK6i48MS/Kb3gtFH63Z2ZVJXwaynYHQuROIva7SR+D24X/IejRdFt9cDGbc3NlSNyDM+qGtH0Wo9/rDY0i0hqHkFEY7ukPRcEcCZpGnbJQO4cbxeVMuM1CcRjf8P4K2i6W+lvdiuj09j8rsIk6lAh6OJR2w2n4VmtNxpd6++/4BWnIgFzhqhtlUnvq9JHeQi6Y09vat26zlmScUO/hnJ98S3fQogE40mzi7TSC2zwkzHSHLf3Ohyjv/n3TgvJXtJd2ouQ8r2F6mifIFz+0+8ZN+LCqTlmA2AznATTfvPfknOuUV+2Am4+iZJL/ZyoUvxz/klZQUkWBPO3yKh8/rddMX/PZVZpYeb4qjLw8pskCHNhU0WYr+RXRPj5w+N9GI+wAxSbPpugzGvW4v7r/wDPufiP"
_raw=zlib.decompress(base64.b64decode(_encoded))
assert hashlib.sha256(_raw).hexdigest()==PAYLOAD_SHA
SOURCES=json.loads(_raw.decode('utf-8'))['sources']
assert len(SOURCES)==24 and len(set(x['problem_id'] for x in SOURCES))==24
assert Counter(x['dataset'] for x in SOURCES)=={'countdown_fresh':8,'gsm8k':8,'svamp':8}
print('Frozen sources:',Counter(x['dataset'] for x in SOURCES))

In [ ]:
# Shared benchmark evaluator. Countdown requires a legal expression with all numbers once.
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
PROMPT_RE=re.compile(r'Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+([-+]?(?:\d+(?:\.\d*)?|\.\d+))',re.I)
ARITH_RE=re.compile(r'[0-9eE+\-*/().=\s]{3,}')

def equals(a,b):
    return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-8)

def lastnum(text):
    x=NUM_RE.findall(str(text).replace(',',''))
    try:return float(x[-1]) if x else None
    except:return None

def eval_arith(expr):
    try:tree=pyast.parse(expr,mode='eval')
    except (SyntaxError,ValueError):return None
    numbers=[]
    def visit(node):
        if isinstance(node,pyast.Expression):return visit(node.body)
        if isinstance(node,pyast.Constant) and type(node.value) in (int,float):
            v=float(node.value)
            if not math.isfinite(v):raise ValueError('nonfinite')
            numbers.append(v);return v
        if isinstance(node,pyast.BinOp) and type(node.op) in (pyast.Add,pyast.Sub,pyast.Mult,pyast.Div):
            a,b=visit(node.left),visit(node.right)
            if isinstance(node.op,pyast.Add):return a+b
            if isinstance(node.op,pyast.Sub):return a-b
            if isinstance(node.op,pyast.Mult):return a*b
            if b==0:raise ZeroDivisionError()
            return a/b
        if isinstance(node,pyast.UnaryOp) and type(node.op) in (pyast.UAdd,pyast.USub):
            a=visit(node.operand);return -a if isinstance(node.op,pyast.USub) else a
        raise ValueError('forbidden AST')
    try:
        ans=visit(tree)
        return (ans,numbers) if math.isfinite(ans) else None
    except (ZeroDivisionError,ValueError,OverflowError):return None

def countdown_check(text,prompt):
    m=PROMPT_RE.search(prompt)
    if not m: raise ValueError('Unexpected Countdown prompt: '+prompt[:200])
    required=[float(x) for x in NUM_RE.findall(m.group(1))];goal=float(m.group(2))
    normalized=str(text).replace('×','*').replace('÷','/').replace('\\times','*')
    for chunk in ARITH_RE.findall(normalized):
        if not any(k in chunk for k in '+-*/'):continue
        for candidate in chunk.split('='):
            candidate=candidate.strip();value=eval_arith(candidate)
            if value is None:continue
            output,nums=value
            if len(nums)==len(required) and all(equals(a,b) for a,b in zip(sorted(nums),sorted(required))) and equals(output,goal):
                return True,candidate
    return False,None

def verify(text,s):
    strict_matches=FINAL_RE.findall(str(text))
    strict=float(strict_matches[-1]) if strict_matches else None
    numeric=lastnum(text)
    if s['dataset']=='countdown_fresh': success,expr=countdown_check(text,s['prompt'])
    else: success=equals(numeric,s['target']);expr=None
    return {'task_success':bool(success),'strict_success':equals(strict,s['target']),
            'numeric_success':equals(numeric,s['target']),'strict_prediction':strict,
            'numeric_prediction':numeric,'valid_expression':expr}

_demo='Use the numbers 19, 29, and 24 exactly once to produce the target 14.'
assert countdown_check('19 + 24 - 29 = 14',_demo)[0]
assert not countdown_check('29 - 19 + 24 = 14',_demo)[0]
assert not countdown_check('29-19=10 FINAL_ANSWER:14',_demo)[0]
print('Verification regression tests PASS')

In [ ]:
MODEL_ID="dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-bd3lm-v0.1"
FAMILY="BD3LM"
RUN_NAME="RepairScopeX_NB19_BD3LM_0p5B"

In [ ]:
# Pin resolved model snapshot and record SHA; no access token needed for public weights.
MODEL_SHA=model_info(MODEL_ID).sha
SNAPSHOT=snapshot_download(repo_id=MODEL_ID,revision=MODEL_SHA)
print('Checkpoint:',MODEL_ID,'snapshot SHA:',MODEL_SHA)
try:
    tokenizer=AutoTokenizer.from_pretrained(SNAPSHOT,trust_remote_code=True)
    model=AutoModelForMaskedLM.from_pretrained(
        SNAPSHOT,trust_remote_code=True,torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    ).to('cuda').eval()
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache(); raise RuntimeError('Model OOM: restart runtime or choose T4/L4 GPU. No silent CPU fallback.')
assert tokenizer.mask_token_id is not None, 'Checkpoint tokenizer lacks mask token.'
MASK_ID=int(tokenizer.mask_token_id)
EOS_ID=tokenizer.eos_token_id
PAD_ID=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else EOS_ID
DEVICE=next(model.parameters()).device
print('Model loaded',type(model).__name__,'dtype',next(model.parameters()).dtype,'mask_id',MASK_ID,'VRAM GB',round(torch.cuda.memory_allocated()/2**30,2))

In [ ]:
OUT_LEN=64
BLOCK=32
STEPS_PER_BLOCK=24
SOURCE_SEEDS=[38191]  # one natural generation per problem. Strictly exploratory.
REPAIR_SEEDS=[38192,38193]
SEED=3819
MAX_REPAIRS=12
PROMPT_SUFFIX='\nReturn a concise answer. For word problems finish with FINAL_ANSWER: <number>. For Countdown give a valid arithmetic expression using each provided number once.'

def prompt_ids(source):
    messages=[{'role':'system','content':'You solve small reasoning and arithmetic problems.'},
              {'role':'user','content':source['prompt']+PROMPT_SUFFIX}]
    ids=tokenizer.apply_chat_template(messages,add_generation_prompt=True,tokenize=True,enable_thinking=False)
    if isinstance(ids,dict):ids=ids['input_ids']
    return [int(t) for t in ids]

def trimmed_tokens(ids):
    # Avoid forcing EOS / padding into a preserved answer; do not re-tokenize decoded text.
    xs=[int(v) for v in ids]
    for k,v in enumerate(xs):
        if v==EOS_ID or v==PAD_ID:
            return xs[:k]
    return xs

def decode(ids):
    return tokenizer.decode([int(i) for i in ids],skip_special_tokens=True).strip()

def set_seed(n):
    torch.manual_seed(int(n));torch.cuda.manual_seed_all(int(n));random.seed(int(n));np.random.seed(int(n)%2**32)

def hash_choice(rows,k):
    return sorted(rows,key=lambda x:hashlib.sha256(f"3824|{x['problem_id']}".encode()).hexdigest())[:k]

# Preserve the exact input-output identity across baseline and repairs.

In [ ]:
ACTIONS=['tail25','tail50','tail75','global']
import copy

def get_transfer(mask,steps):
    n=int(mask.sum().item())
    base,rem=divmod(n,steps)
    return [base+(j<rem) for j in range(steps)]

def staircase(x,block_size,pad_id):
    B,T=x.shape
    valid=x!=pad_id
    pos_raw=valid.long().cumsum(-1)
    pos_ids=torch.where(valid,pos_raw-1,torch.zeros_like(pos_raw)).long()
    col=torch.arange(T,device=x.device)
    ids=(col//block_size).view(1,T).expand(B,T)
    ids=torch.where(valid,ids,torch.full_like(ids,-1))
    q=ids.view(B,1,T,1);k=ids.view(B,1,1,T)
    attn=(k<=q)&(q>=0)&(k>=0)
    return attn,pos_ids

@torch.inference_mode()
def denoise(src_ids,start=None,action='global',seed=38191):
    set_seed(seed)
    prompt=list(src_ids)
    if start is None:
        preserved=[];budget=OUT_LEN;original=[]
    else:
        original=list(start);L=len(original)
        assert L>=8
        if action=='global':cut=0
        elif action=='tail25':cut=max(1,int(.75*L))
        elif action=='tail50':cut=max(1,int(.50*L))
        elif action=='tail75':cut=max(1,int(.25*L))
        else:raise ValueError(action)
        preserved=original[:cut]
        budget=L-cut
    x=torch.tensor([prompt+preserved],dtype=torch.long,device=DEVICE)
    plen=len(prompt);remain=budget;calls=0
    t0=time.perf_counter();torch.cuda.reset_peak_memory_stats()
    while remain>0:
        old_len=x.size(1)
        offset=old_len%BLOCK
        room=BLOCK if offset==0 else BLOCK-offset
        cur_len=min(room,remain)
        a,p=staircase(x,BLOCK,PAD_ID)
        pref=model(x,attention_mask=a,position_ids=p,use_cache=True)
        calls+=1
        past=pref.past_key_values
        block=torch.full((1,cur_len),MASK_ID,dtype=torch.long,device=DEVICE)
        x=torch.cat([x,block],dim=1)
        attn_all,pos_all=staircase(x,BLOCK,PAD_ID)
        attn_blk=attn_all[:,:,old_len:,:]
        pos_blk=pos_all[:,old_len:]
        for planned in get_transfer(block,STEPS_PER_BLOCK):
            cur=x[:,old_len:]
            mask=cur[0]==MASK_ID
            if not mask.any():break
            if planned==0:continue
            logits=model(cur,attention_mask=attn_blk,position_ids=pos_blk,
                         past_key_values=copy.deepcopy(past),use_cache=False).logits
            calls+=1
            lg=logits[0];guess=lg.argmax(-1)
            score=lg.gather(1,guess[:,None])[:,0]-torch.logsumexp(lg,dim=-1)
            score=score.masked_fill(~mask,-float('inf'))
            idx=torch.topk(score,k=min(int(mask.sum().item()),planned)).indices
            x[0,old_len+idx]=guess[idx]
        assert not (x[0,old_len:]==MASK_ID).any(),'Block not fully committed'
        remain-=cur_len
    all_ids=x[0].tolist()
    new_ids=all_ids[plen+len(preserved):]
    combined=preserved+new_ids
    assert combined[:len(preserved)]==preserved
    wall=time.perf_counter()-t0
    return {'ids':combined,'nfe':calls,'wall_seconds':wall,'peak_vram_gb':float(torch.cuda.max_memory_allocated()/2**30),
            'forced_tokens':len(preserved),'answer_canvas_tokens':len(combined),'scope':action}

In [ ]:
from google.colab import drive,files
SAVE_TO_DRIVE=True
try:
    if SAVE_TO_DRIVE: drive.mount('/content/drive')
    OUT=Path('/content/drive/MyDrive/RepairScopeX/'+RUN_NAME) if SAVE_TO_DRIVE else Path('/content/'+RUN_NAME)
except Exception as ex:
    print('Drive mount failed, using /content, resume will not survive runtime disconnection:',ex)
    OUT=Path('/content/'+RUN_NAME)
OUT.mkdir(exist_ok=True,parents=True)
MAN=OUT/'manifest.json';GEN=OUT/'natural_generations.jsonl';REPAIR=OUT/'repair_trials.jsonl'
config={'model':MODEL_ID,'sha':MODEL_SHA,'family':FAMILY,'payload_sha':PAYLOAD_SHA,
        'source_ids':[x['problem_id'] for x in SOURCES], 'out_len':OUT_LEN,
        'block':BLOCK,'steps_per_block':STEPS_PER_BLOCK,'source_seeds':SOURCE_SEEDS,
        'repair_seeds':REPAIR_SEEDS,'actions':ACTIONS,'max_repairs':MAX_REPAIRS}
sig=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
if MAN.exists():
    old=json.loads(MAN.read_text())
    assert old['signature']==sig,'Existing Drive results use different parameters: set a new RUN_NAME.'
else: MAN.write_text(json.dumps({'signature':sig,'config':config,'gpu':torch.cuda.get_device_name(0),'created_utc':datetime.now(timezone.utc).isoformat()},indent=2))

def load_lines(p):
    ans={}
    if not p.exists():return ans
    with p.open(encoding='utf-8') as f:
        for n,line in enumerate(f,1):
            if not line.strip():continue
            try:r=json.loads(line)
            except json.JSONDecodeError as e:raise RuntimeError(f'Corrupt JSONL {p.name} line {n}') from e
            assert r['signature']==sig,'Run-signature conflict'
            assert r['id'] not in ans,'Duplicate trial ID'
            ans[r['id']]=r
    return ans

def append(path,row):
    with path.open('a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+'\n');f.flush();os.fsync(f.fileno())

def run_source(src,seed):
    result=denoise(prompt_ids(src),seed=seed)
    ids=trimmed_tokens(result['ids'])
    output=decode(ids)
    res=verify(output,src)
    return {'id':src['problem_id']+'|'+str(seed),'problem_id':src['problem_id'], 'dataset':src['dataset'],
            'prompt':src['prompt'],'target':src['target'],'seed':seed,'token_ids':ids,
            'output':output,'verification':res,'task_success':res['task_success'],
            'strict_success':res['strict_success'],'numeric_success':res['numeric_success'],
            'nfe':result['nfe'],'wall_seconds':result['wall_seconds'],'peak_vram_gb':result['peak_vram_gb']}

def run_repair(src,failure,action,seed):
    result=denoise(prompt_ids(src),start=failure['token_ids'],action=action,seed=seed)
    ids=result['ids'];output=decode(ids);res=verify(output,src)
    orig=failure['token_ids'];preservation=sum(a==b for a,b in zip(orig,ids))/max(len(orig),len(ids),1)
    pred0=failure['verification']['numeric_prediction'];pred1=res['numeric_prediction']
    return {'id':failure['id']+'|'+action+'|'+str(seed),'failure_id':failure['id'],'problem_id':src['problem_id'],
            'dataset':src['dataset'],'action':action,'repair_seed':seed,'output':output,'token_ids':ids,
            'task_success':res['task_success'],'strict_success':res['strict_success'],
            'numeric_success':res['numeric_success'],'verification':res,
            'same_wrong_numeric':bool(not res['task_success'] and pred0 is not None and pred1 is not None and equals(pred0,pred1)),
            'nfe':result['nfe'],'wall_seconds':result['wall_seconds'],'peak_vram_gb':result['peak_vram_gb'],
            'forced_tokens':result['forced_tokens'],'answer_canvas_tokens':result['answer_canvas_tokens'],
            'token_position_preservation':preservation,'forced_fraction':result['forced_tokens']/max(1,result['answer_canvas_tokens'])}

In [ ]:
# 1. One smoke generation before full sweep; every row is persisted immediately.
source_by_id={x['problem_id']:x for x in SOURCES}
smoke=run_source(SOURCES[0],SOURCE_SEEDS[0]);print('SMOKE',smoke['id'],'NFE',smoke['nfe'],'task',smoke['task_success'],'output:',smoke['output'][:250])
assert smoke['nfe']>0 and len(smoke['token_ids'])>0
print('REAL MODEL SMOKE PASS')

done=load_lines(GEN)
for i,s in enumerate(SOURCES,1):
    for seed in SOURCE_SEEDS:
        tid=s['problem_id']+'|'+str(seed)
        if tid in done:continue
        try:
            r=run_source(s,seed);r.update(status='ok',error=None,signature=sig)
        except Exception as e:
            r={'id':tid,'problem_id':s['problem_id'],'dataset':s['dataset'],'status':'error',
               'error':f'{type(e).__name__}: {e}','signature':sig}
            append(GEN,r)
            raise RuntimeError(f'Generation {tid} failed; result saved, fix and re-run cell') from e
        append(GEN,r)
        print('GEN',i,'/',len(SOURCES),tid,'NFE',r['nfe'],'OK',r['task_success'])

naturals=load_lines(GEN)
assert len(naturals)==len(SOURCES)*len(SOURCE_SEEDS)
assert all(r['status']=='ok' for r in naturals.values())
failures=[r for r in naturals.values() if not r['task_success'] and len(r['token_ids'])>=8]
# A deterministic, pre-repair-balanced failure selection (no outcome leakage).
chosen=[]
for ds in ('countdown_fresh','gsm8k','svamp'):
    dsf=[r for r in failures if r['dataset']==ds]
    chosen+=sorted(dsf,key=lambda r:hashlib.sha256(('3824|'+r['problem_id']).encode()).hexdigest())[:4]
left=[r for r in failures if r['id'] not in {x['id'] for x in chosen}]
chosen+=(sorted(left,key=lambda r:hashlib.sha256(('3824|'+r['problem_id']).encode()).hexdigest())[:MAX_REPAIRS-len(chosen)])
chosen=chosen[:MAX_REPAIRS]
print('Natural failures:',len(failures),' / ',len(naturals),'Selected',len(chosen),'by dataset',Counter(x['dataset'] for x in chosen))
assert len(chosen)==len(set(x['id'] for x in chosen))
# Build frozen cohort record with hash: if results exist, don't silently change cohort.
cohort_file=OUT/'selected_failures.json'
cohort_ids=[r['id'] for r in chosen]
if cohort_file.exists():
    assert json.loads(cohort_file.read_text())==cohort_ids,'Different failure selection versus saved run'
else:cohort_file.write_text(json.dumps(cohort_ids,indent=2))

completed=load_lines(REPAIR)
for i,fail in enumerate(chosen,1):
    src=source_by_id[fail['problem_id']]
    for action in ACTIONS:
        for seed in REPAIR_SEEDS:
            tid=fail['id']+'|'+action+'|'+str(seed)
            if tid in completed:continue
            try:r=run_repair(src,fail,action,seed);r.update(status='ok',error=None,signature=sig)
            except Exception as e:
                r={'id':tid,'failure_id':fail['id'],'action':action,'repair_seed':seed,
                   'status':'error','error':f'{type(e).__name__}: {e}','signature':sig}
                append(REPAIR,r);raise RuntimeError('Repair trial failed; saved error before exit: '+tid) from e
            append(REPAIR,r)
            print('REPAIR',i,'/',len(chosen),action,seed,'success',r['task_success'],'NFE',r['nfe'])
print('Sweeps complete')

In [ ]:
!pip -q install scipy

In [ ]:
from scipy.stats import binomtest
natural=load_lines(GEN);trials=load_lines(REPAIR)
expected=len(chosen)*len(ACTIONS)*len(REPAIR_SEEDS)
assert len(trials)==expected,(len(trials),expected)
assert all(r['status']=='ok' for r in trials.values())
DF=pd.DataFrame(trials.values());ND=pd.DataFrame(natural.values())
print('Natural generations:',len(ND),'genuine failures',int((~ND['task_success']).sum()),
      'repair trials:',len(DF),'repair successes:',int(DF['task_success'].sum()))
assert len(DF['id'].unique())==len(DF)
assert (DF['token_position_preservation']+1e-9>=DF['forced_fraction']).all()
ACT=(DF.groupby('action',as_index=False)
     .agg(trials=('task_success','size'),success_rate=('task_success','mean'),strict_success=('strict_success','mean'),
          numeric_success=('numeric_success','mean'),mean_preservation=('token_position_preservation','mean'),
          mean_forced_fraction=('forced_fraction','mean'),mean_nfe=('nfe','mean'),mean_wall_seconds=('wall_seconds','mean'),
          same_wrong_rate=('same_wrong_numeric','mean')))
DATA=(DF.groupby(['dataset','action'],as_index=False)
      .agg(n=('task_success','size'),success_rate=('task_success','mean'),mean_preservation=('token_position_preservation','mean')))
print('\\nAction summary:');display(ACT);print('\\nDataset summary:');display(DATA)
# Cluster-bootstrap resamples problems, retaining the multiple repair seeds together.
rng=np.random.default_rng(1419);BOOT=[]
for act in ACTIONS:
    sub=DF[DF['action']==act]
    group=sub.groupby('problem_id')['task_success'].mean()
    vals=group.to_numpy(dtype=float)
    boots=np.mean(rng.choice(vals,size=(3000,len(vals)),replace=True),axis=1)
    BOOT.append(dict(action=act,problems=len(vals),success=float(sub['task_success'].mean()),ci_low=float(np.quantile(boots,.025)),ci_high=float(np.quantile(boots,.975))))
CI=pd.DataFrame(BOOT)
print('Cluster CIs');display(CI)
# A paired per-problem (not per-seed) exact sign test against global.
PIV=DF.groupby(['problem_id','action'])['task_success'].mean().unstack()
TEST=[]
for act in ACTIONS:
    if act=='global':continue
    diff=(PIV[act]-PIV['global']).dropna()
    wins=int((diff>0).sum());losses=int((diff<0).sum());d=wins+losses
    TEST.append(dict(action=act,problem_count=len(diff),mean_difference=float(diff.mean()),wins=wins,losses=losses,
                     p_two_sided=float(binomtest(wins,d,p=.5).pvalue) if d else 1.0))
TEST=pd.DataFrame(TEST)
print('Problem-level tests');display(TEST)
# In-sample oracle is optimistic and NOT a deployable selection policy.
PER=DF.groupby(['problem_id','dataset','action'],as_index=False)['task_success'].mean()
oracle=float(PER.groupby('problem_id')['task_success'].max().mean()) if len(PER) else 0.
fixed=float(ACT['success_rate'].max()) if len(ACT) else 0.
GATE={'model':MODEL_ID,'family':FAMILY,'model_revision':MODEL_SHA,'natural_count':len(ND),
      'natural_task_failures':int((~ND['task_success']).sum()),'selected_failure_count':len(chosen),
      'repair_trials':len(DF),'repair_successes':int(DF['task_success'].sum()),
      'success_datasets':int(DF.loc[DF['task_success'],'dataset'].nunique()),
      'in_sample_oracle_success':oracle,'best_fixed_success':fixed,'in_sample_orh':oracle-fixed,
      'safety_preservation_audit':bool((DF['token_position_preservation']+1e-9>=DF['forced_fraction']).all()),
      'complete':len(ND)==24 and len(DF)==expected,'caution':'Small coder-specialized 0.5B research checkpoints; exploratory, not a publication-ready generalization claim.'}
for df,path in [(ACT,'action_summary.csv'),(DATA,'dataset_summary.csv'),(CI,'cluster_ci.csv'),(TEST,'paired_problem_tests.csv'),(PER,'per_problem_surface.csv')]:df.to_csv(OUT/path,index=False)
(OUT/'GATE.json').write_text(json.dumps(GATE,indent=2))
print(json.dumps(GATE,indent=2))

In [ ]:
import shutil
manifest={'signature':sig,'model_sha':MODEL_SHA,'generated_utc':datetime.now(timezone.utc).isoformat(),'files':{}}
for f in sorted(OUT.glob('*')):
    if f.is_file() and f.name!='artifact_manifest.json':
        manifest['files'][f.name]=hashlib.sha256(f.read_bytes()).hexdigest()
(OUT/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2))
zip_path=shutil.make_archive('/content/'+RUN_NAME+'_results','zip',root_dir=OUT)
print('RESULT ZIP:',zip_path,'bytes:',Path(zip_path).stat().st_size)
files.download(zip_path)

### Result export
Send back the automatically downloaded `RepairScopeX_NB19_BD3LM_0p5B_results.zip` file. The `GATE.json` and raw JSONL logs are essential. If the model fails to load, send the exception rather than treating the experiment as successful.